In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import uproot
import sys
import math

dir = "/Users/alexanderantonakis/Desktop/Software/AFrameAnalysis/Macros/"

sys.path.append("../Utils")
sys.path.append("../Configs")

from ChannelMap import ChannelMap

frame = 1
strip_w = 11.2 # cm 
#cluster_file = "../OfficialClusters/clusters_frame7.root"
cluster_file = "clusters_frame1.root"
config = "config_frame1.txt"
run_config = "run_config_frame1.txt"

fig_dir = "../Figs/Frame"+str(frame)+"/"

# set up the geometry of the frame
map = ChannelMap("../Configs/"+config, "../Configs/"+run_config)
map.initialize_config()
map.initialize_run_config()
map.calculate_params()
print("initialized the geometry and voltages")
print("")

# initialize the horizontal febs --> useful to have
febs = map.mac5
horiz_febs = []
for feb in febs:
    if map.is_horiz(feb):
        horiz_febs.append(feb)
        
print("All Horizontal FEBs in this file:", horiz_febs)


# Open the ROOT file and the TTree
file = uproot.open(cluster_file)  # Replace with your ROOT file
tree = file["cluster_tree"]    

# Convert the TTree to a pandas DataFrame
df = tree.arrays(library="pd")

df[:2]

In [ ]:
# Make DataFrames that are easier to work with
strip_df = pd.DataFrame(df['strips'].tolist())
columns = ["Run"]
for feb in horiz_febs:
    columns.append("FEB"+str(feb))
strip_df.columns = columns

print("made the strip dataframe")
strip_df[:5]

In [ ]:
# make a run string for saving plots
all_runs_header = list(set(list(strip_df["Run"].values)))
all_runs_header.sort()
run_str = "_runs:"
count = 0
for num in all_runs_header:
    run_str+=str(num)
    if count != len(all_runs_header) - 1:
        run_str+=":"
    count += 1
run_str += "_"
print("Run String", run_str)

In [ ]:
time_df = pd.DataFrame(df['times'].tolist())

time_df.columns = columns

print("made the time dataframe")
time_df[:5]

In [ ]:
adcA_df = pd.DataFrame(df['adcA'].tolist())

adcA_df.columns = columns

print("made the adcA dataframe")
adcA_df[:5]

In [ ]:
adcB_df = pd.DataFrame(df['adcB'].tolist())

adcB_df.columns = columns

print("made the adcB dataframe")
adcB_df[:5]

In [ ]:
runs = list(set(list(strip_df['Run'].values)))
print("runs", runs)

In [ ]:
def deltaTCut(row):
    t_vals = []
    for feb in horiz_febs:
        t = row["FEB"+str(feb)]
        if t != -1:
            t_vals.append(t)
    dt = max(t_vals) - min(t_vals)
    return dt
    
time_df['dtCut'] = time_df.apply(deltaTCut, axis=1)

plt.hist(time_df["dtCut"].values, bins=100, color="r")
plt.xlabel(r"$\Delta$t Cut Variable [ns]")
plt.ylabel("Counts")
plt.title("Frame "+str(frame)+" All Cluster Candidates")
plt.yscale("log")
#plt.savefig(fig_dir+"frame"+str(frame)+run_str+"delTCut_all_"+str(int(100*goal))+".png", 
#            format='png')
plt.show()

In [ ]:
feb_h_indices = [map.mac5.index(horiz_febs[num]) for num in range(len(horiz_febs))]
angles_h = [map.angle[feb_h_indices[num]] for num in range(len(feb_h_indices))] 

# apply uniform errors
sigma = strip_w/np.sqrt(12.0)

# errors for each horizontal module
all_err_z = [sigma*np.cos(angles_h[num]) for num in range(len(angles_h))]
all_err_y = [sigma*np.sin(angles_h[num]) for num in range(len(angles_h))]

print("calculated uniform hit errors")

In [ ]:
def calculate_z(row):
    z_vals = []
    for feb in horiz_febs:
        strip = row["FEB"+str(feb)]
        if strip == -1:
            # give an absurd value
            z_vals.append(-999.)
        else:
            idx = map.mac5.index(feb)
            zpos = map.zpos[idx]
            angle = map.angle[idx]
            direction = map.channel_ascending[idx]
            hypotenuse1 = strip*strip_w + (strip_w/2)
            hypotenuse2 = 16*strip_w - strip*strip_w
            if zpos < 0:
                if direction:
                    z_vals.append(np.cos(angle)*hypotenuse1 + zpos)
                else:
                    z_vals.append(np.cos(angle)*hypotenuse2 + zpos)         
            else:
                if direction:
                    z_vals.append(zpos - np.cos(angle)*hypotenuse1)
                else:
                    z_vals.append(zpos - np.cos(angle)*hypotenuse2)
    return z_vals

strip_df['z'] = strip_df.apply(calculate_z, axis=1)
strip_df[:4]

In [ ]:
z_df = pd.DataFrame(strip_df['z'].tolist())

z_df.columns = columns[1:]
z_df['Run'] = strip_df['Run']
z_df[:4]

In [ ]:
def calculate_y(row):
    y_vals = []
    for feb in horiz_febs:
        strip = row["FEB"+str(feb)]
        if strip != -1:
            idx = map.mac5.index(feb)
            angle = map.angle[idx]
            direction = map.channel_ascending[idx]
            hypotenuse1 = strip*strip_w + (strip_w/2)
            hypotenuse2 = 16*strip_w - strip*strip_w
            if direction:
                y_vals.append(hypotenuse1*np.sin(angle))
            else:
                y_vals.append(hypotenuse2*np.sin(angle))
        else:
            y_vals.append(-999.0)
            
    return y_vals
    
strip_df['y'] = strip_df.apply(calculate_y, axis=1)
strip_df[:4]

In [ ]:
y_df = pd.DataFrame(strip_df['y'].tolist())

y_df.columns = columns[1:]
y_df['Run'] = strip_df['Run']
y_df[:4]

In [ ]:
import ROOT

def line_fit(row1, row2):
    N = 0
    g0 = ROOT.TGraphErrors()
    for feb in horiz_febs:
        z = row1["FEB"+str(feb)]
        y = row2["FEB"+str(feb)]
        if z > -990.0:
            idx = horiz_febs.index(feb)
            g0.SetPoint(N, z, y)
            g0.SetPointError(N, all_err_z[idx], all_err_y[idx])
            N += 1

    # Define a linear function (y = mx + b)
    linear_func = ROOT.TF1("linear_func", "[0]*x + [1]")
    dx = g0.GetPointX(0) - g0.GetPointX(N-1)
    dy = g0.GetPointY(0) - g0.GetPointY(N-1)
    
    m_start = 1.0
    if dx != 0.0:
        m_start = dy/dx

    b0 = g0.GetPointY(N-1) - m_start*g0.GetPointX(N-1)
    b1 = g0.GetPointY(0) - m_start*g0.GetPointX(0)    
    b_start = (b0 + b1)/2.0
    
    # Set some initial parameters for the fit
    linear_func.SetParameter(0, m_start)
    linear_func.SetParameter(1, b_start)

    # Fit the graph with the linear function
    #g0.Fit("linear_func", "", "BRQE")
    g0.Fit("linear_func", "Q", "")
    # Get the fit results
    fit_results0 = g0.GetFunction("linear_func")

    m0, b0 = fit_results0.GetParameter(0), fit_results0.GetParameter(1)    
    m0_err, b0_err = fit_results0.GetParError(0), fit_results0.GetParError(1)
    chis = g0.Chisquare(g0.GetFunction("linear_func"))
        
    return [m0, b0, m0_err, b0_err, chis, N]



# Use zip to combine rows from both DataFrames and apply the function
result = z_df.apply(lambda row1: line_fit(row1, y_df.loc[row1.name]), axis=1)

fit_df = pd.DataFrame(result.tolist(), columns=['m', 'b', 'm_err', 'b_err', 'chis', 'N'])
fit_df[:4]

In [ ]:
import math
# Let's add the zenith angle
def get_theta(row):
    m = row['m']
    theta = ((math.pi/2) - np.arctan(abs(m)))*(180/math.pi)
    return theta

fit_df['theta'] = fit_df.apply(get_theta, axis=1)
fit_df[:4]

In [ ]:
chis_per_N_vals = np.linspace(1, 6, 12)
theta_bins = np.linspace(0, 90, 20)

plt.hist(fit_df['theta'].values, bins=theta_bins, histtype="step", label = "All Clusters")
for num in chis_per_N_vals:
    plt.hist(fit_df.query("chis/N < @num")['theta'].values, 
             bins=theta_bins, histtype="step", label=r"$\chi^{2}/dof$ < "+str(round(num, 1)))
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.xlabel("Zenith Angle [degrees]", fontsize=14)
plt.ylabel("Counts", fontsize=14)
plt.title("All Voltages: Frame "+str(frame), fontsize=16)
#plt.savefig(fig_dir+"frame"+str(frame)+run_str+"zenith_all_"+str(int(100*goal))+".png", 
#                format='png', bbox_inches='tight')
plt.show()

In [ ]:
fit_df["chi_dof"] = fit_df["chis"]/fit_df["N"]
fit_df[:4]

In [ ]:
from matplotlib.colors import LogNorm

plt.hist2d(fit_df['theta'].values, 
           fit_df['chis'].values / fit_df['N'].values, bins=100, cmap="plasma", norm=LogNorm())

plt.colorbar(label='Counts')
plt.xlabel("Zenith Angle [degrees]", fontsize=14)
plt.ylabel(r"$\chi^2$/dof", fontsize=14)
plt.title("All Cluster Candidates", fontsize=16)
#plt.savefig(fig_dir+"frame"+str(frame)+run_str+"chi_angle_2d_all"+"_goal"+str(int(100*goal))+".png", 
#                format='png')
plt.show()

In [ ]:
# Let's look at the chis vs time cut


fit_df["dtCut"] = time_df["dtCut"]


plt.hist2d(fit_df.query("chi_dof < 20")['dtCut'].values, 
           fit_df.query("chi_dof < 20")['chi_dof'].values, bins=100, cmap="plasma", norm=LogNorm())

plt.colorbar(label='Counts')
plt.xlabel(r"$\Delta$t Cut Variable [ns]", fontsize=14)
plt.ylabel(r"$\chi^2$/dof", fontsize=14)
plt.title("All Cluster Candidates", fontsize=16)
#plt.savefig(fig_dir+"frame"+str(frame)+run_str+"chi_angle_2d_all"+"_goal"+str(int(100*goal))+".png", 
#                format='png')
plt.show()


# New Diagnostics --> Ready for Tracks!

In [ ]:
# First, let's look at basic hit metrics

fig, axs = plt.subplots(3, 4, figsize=(12, 6))
count = 0
i = 0
j = 0
for feb in horiz_febs:  
    for run in runs:
        #if run == 5806:
        #    continue
        axs[i,j].hist(strip_df.query("Run == "+str(run) + " and FEB"+str(feb)+" > -1")["FEB"+str(feb)].values, 
                 bins=16, histtype="step", label="Run "+str(run))
        
    
    axs[i, j].set_title("FEB "+str(feb))
    if j == 3:
        axs[i, j].legend(bbox_to_anchor=(1, 0.5))
    
    axs[i, j].set_yscale('log')
    j += 1    
    if j > 3:
        j = 0
        i += 1
    
  
#fig.suptitle("Frame "+str(frame)+ " Run " + str(run) + " Voltage Setting "+ str(voltage), fontsize=16)
fig.text(0.5, -0.02, 'Strip ID', ha='center', fontsize=20)
fig.text(-0.02, 0.5, '300 ns Cluster Counts', ha='center', rotation='vertical', fontsize=20)
plt.tight_layout()
#plt.savefig(fig_dir+"frame"+str(frame)+run_str+"reco_rates_volt"+str(voltage)+"_"+str(int(100*goal))+".png", 
#            format='png')
plt.show()
          

In [ ]:
def was_hit(m, b, feb):
    index = map.mac5.index(feb)
    zint = (map.b[index] - b) /(m - map.m[index])
    yint = m*zint + b
    if 0 <= yint <= map.ymax[index]:
        return 1
    else:
        return 0


def get_hit(m, b, feb):
    index = map.mac5.index(feb)
    zint = (map.b[index] - b) /(m - map.m[index])
    #yint = m*zint + b
    return zint


def plot_track(rowz, rowfit, geo):

    def line(x, m, b):
        return m*x + b

    def get_y(z_hit, z_module, angle):
        dz = abs(z_module - z_hit)
        y = np.tan(angle)*dz
        return y
        
    #ind = [i for i in range(len(horiz_febs)) if row[str(horiz_febs[i])] != 0]

    m = rowfit['m']
    b = rowfit['b']
    z_vals = []
    y_vals = []
    y_err = []
    z_err = []
    was_m_hit = [0 for num in horiz_febs]
    bad_z = []
    bad_y = []
    for feb in horiz_febs:
        i = map.mac5.index(int(feb))
        zpos = map.zpos[i]
        angle = map.angle[i]
        if rowz["FEB"+str(feb)] < -900:
            if was_hit(m,b,feb):
                z_hit = get_hit(m, b, feb)
                y_hit = get_y(z_hit, zpos, angle)
                bad_z.append(z_hit)
                bad_y.append(y_hit)
                
            continue
        z_vals.append(rowz["FEB"+str(feb)])
        #i = map.mac5.index(int(feb))
        was_m_hit[i-2] = 1
        #zpos = map.zpos[i]
        #angle = map.angle[i]
        y_vals.append(get_y(rowz["FEB"+str(feb)], zpos, angle))

        idx = feb_h_indices.index(i)
        y_err.append(all_err_y[idx])
        z_err.append(all_err_z[idx])
         
    z_fit = np.linspace(min(z_vals)-1, max(z_vals)+1, 1000)
    y_fit = line(z_fit, m, b)

    z_modules = []
    y_modules = []
    for num in range(len(horiz_febs)):
        i = map.mac5.index(horiz_febs[num])
        zpos = map.zpos[i]
        angle = map.angle[i]
        zmax = -1.0
        ymax = np.sin(angle)*map.module_width_h
        if zpos < 0:
            zmax = np.cos(angle)*map.module_width_h + zpos
            y_modules.append([0, ymax])
            z_modules.append([zpos, zmax])
            
        else:
            #print("zpos > 0", zpos)
            zmax = zpos - np.cos(angle)*map.module_width_h 
            #print("zmax", zmax)
            y_modules.append([ymax, 0])
            z_modules.append([zmax, zpos])
        
    
    #print("about to make the plot")
    plt.scatter(z_vals, y_vals, c="r", label="module hits")
    plt.plot(z_fit, y_fit, label=r"$\chi^2$ = "+str(round(rowfit['chis'], 2)) + r",  $\theta$ ="+ str(round(rowfit['theta'], 2)))
    plt.errorbar(z_vals, y_vals, yerr=y_err, xerr=z_err, fmt='', ecolor=None, elinewidth=None,
                 capsize=None, barsabove=False, lolims=False, uplims=False, xlolims=False,
                 xuplims=False, errorevery=1, capthick=None, ls='none', color='k')
    count = 0
    if geo:
        for num in range(len(bad_z)):
            plt.scatter(bad_z[num], bad_y[num], c="purple")
            
        for num in range(len(z_modules)):
            color = "green"
            if was_m_hit[num]:
                color = "orange"
            if count == 0:
                plt.plot(z_modules[num], y_modules[num], c=color, label="modules")
                count += 1
            else:
                plt.plot(z_modules[num], y_modules[num], c=color)
    plt.title("Frame "+str(frame) + " Run "+str(int(rowz["Run"])))            
    plt.xlabel("Reconstructed Z [cm]")
    plt.ylabel("Reconstructed Y [cm]")

    plt.legend()
    plt.show()

In [ ]:
tot_df = pd.concat([z_df, fit_df], axis=1)
tot_df[:4]    

In [ ]:

#tot_df = tot_df.query("15 < chi_dof < 20")
sel = "Run == 6106 and chi_dof < 1.5 and theta < 85 and dtCut < 75"
for num in range(10):
    plot_track(tot_df.query(sel).iloc[num], tot_df.query(sel).iloc[num], 1)
    #plot_track(tot_df.iloc[num], tot_df.iloc[num], 1)

In [ ]:




plt.hist2d(tot_df.query("FEB153 > -990")['dtCut'].values, 
           tot_df.query("FEB153 > -990")['chi_dof'].values, bins=100, cmap="plasma", norm=LogNorm())

plt.colorbar(label='Counts')
plt.xlabel(r"$\Delta$t Cut Variable [ns]", fontsize=14)
plt.ylabel(r"$\chi^2$/dof", fontsize=14)
#plt.title("", fontsize=16)
#plt.savefig(fig_dir+"frame"+str(frame)+run_str+"chi_angle_2d_all"+"_goal"+str(int(100*goal))+".png", 
#                format='png')
plt.show()

In [ ]:
fit_df["Run"] = z_df["Run"]

for num in range(10):
    plot_track(z_df.query("Run == 5832").iloc[num], fit_df.query("Run == 5832").iloc[num], 1)

In [ ]:
import matplotlib.cm as cm
from scipy.optimize import curve_fit
from scipy import special
import math


def eff_func(x, a, b, c):
        return a*special.erf(x-c, out=None) + b



v_sort = (66.5156862745098, 66.82941176470588, 67.3)
e_sort = (0.8815142576204523, 0.9254601226993865, 0.6583805070145213)
err_sort = (0.0050938630124883755, 0.004619082655806239, 0.005)

#v_trunc (66.5156862745098, 66.82941176470588)
#e_trunc (0.8815142576204523, 0.9254601226993865)
#err_trunc (0.0050938630124883755, 0.004619082655806239)

popt, pcov = curve_fit(eff_func, v_sort[:-1], e_sort[:-1], p0=[1.0, 0.5, 67.0], sigma=err_sort[:-1])


x = np.linspace(min(v_sort), max(v_sort), 1000)
y = eff_func(x, popt[0], popt[1], popt[2])

plt.scatter(v_sort, e_sort)
plt.plot(x, y)
plt.show()